# Phase 6 — Feature Selection and Encoding

Goals:
- run selected feature selection method
- identify redundant features
- check multicollinearity
- finalize regression and clustering inputs
- apply encoding per modeling path
- save preprocessors and model-input datasets

# Imports

In [13]:
import joblib
import pandas as pd
import numpy as np

from src.utils.paths import find_project_root, resolve_project_path
from src.utils.io import read_csv_file, save_csv_file, save_text_file
from src.utils.config import load_project_configs

from src.features.feature_selection import (
    select_regression_features,
    select_clustering_features,
    plot_feature_importance,
    plot_correlation_heatmap_for_selected,
    plot_vif_summary,
)
from src.features.encoding import (
    build_regression_preprocessor,
    build_clustering_preprocessor,
    transform_with_preprocessor,
)
from src.features.scaling import fit_numeric_scaler
from src.features.build_features import build_feature_engineering_report

# Load configs and engineered data

In [2]:
project_root = find_project_root()
configs = load_project_configs(project_root)
features_cfg = configs["features_config"]
main_cfg = configs["main_config"]

df_fe = read_csv_file(resolve_project_path(project_root, "data/processed/diamonds_feature_engineered.csv"))
df_fe.shape

(53940, 22)

## Define feature groups

In [3]:
raw_numeric_cols = features_cfg["schema"]["numeric_features"]
raw_categorical_cols = features_cfg["schema"]["categorical_features"]

engineered_numeric_cols = [
    "volume",
    "volume_proxy",
    "dimension_ratio",
    "length_width_ratio",
    "depth_pct_from_dimensions",
    "table_depth_interaction",
    "carat_squared",
    "table_to_depth_ratio",
    "face_area",
]
engineered_categorical_cols = ["carat_category"]

## Regression feature selection

In [4]:
regression_selection = select_regression_features(
    df = df_fe,
    raw_numeric_cols = raw_numeric_cols,
    raw_categorical_cols = raw_categorical_cols,
    engineered_numeric_cols = engineered_numeric_cols,
    engineered_categorical_cols = engineered_categorical_cols,
    target_col = "price",
    importance_threshold = 0.01,
    corr_threshold = 0.90,
    vif_threshold = 10.0,
    random_state = main_cfg["project"]["random_state"],
)

regression_selection["selected_features"]

['volume', 'carat', 'depth', 'table', 'clarity', 'color', 'cut']

# Clustering feature selection

In [5]:
clustering_selection = select_clustering_features(
    df = df_fe,
    raw_numeric_cols = raw_numeric_cols,
    raw_categorical_cols = raw_categorical_cols,
    engineered_numeric_cols = engineered_numeric_cols,
    engineered_categorical_cols = engineered_categorical_cols,
    corr_threshold = 0.90,
)

clustering_selection["selected_features"]

['carat',
 'depth',
 'table',
 'dimension_ratio',
 'length_width_ratio',
 'depth_pct_from_dimensions',
 'table_depth_interaction',
 'table_to_depth_ratio',
 'cut',
 'color',
 'clarity']

## Save feature selection figures

In [6]:
plot_feature_importance(
    regression_selection["importance_df"],
    resolve_project_path(project_root, "figures/feature_engineering/feature_importance_baseline.png"),
)

plot_correlation_heatmap_for_selected(
    df_fe,
    regression_selection["selected_numeric_features"],
    resolve_project_path(project_root, "figures/feature_engineering/correlation_selected_features.png"),
)

plot_vif_summary(
    regression_selection["vif_df"],
    resolve_project_path(project_root, "figures/feature_engineering/vif_summary_plot.png"),
)

print("Saved feature-selection figures.")

Saved feature-selection figures.


## Build regression and clustering datasets

In [7]:
regression_dataset = df_fe[regression_selection["selected_features"] + ["price"]].copy()
clustering_dataset = df_fe[clustering_selection["selected_features"]].copy()

# Deliverable rule: clustering must not include price
clustering_dataset = clustering_dataset.drop(columns=["price"], errors="ignore")

regression_dataset.shape, clustering_dataset.shape

((53940, 8), (53940, 11))

## Save model-input datasets

In [8]:
regression_path = resolve_project_path(project_root, "data/processed/regression_model_input.csv")
clustering_path = resolve_project_path(project_root, "data/processed/clustering_model_input.csv")

save_csv_file(regression_dataset, regression_path, index=False)
save_csv_file(clustering_dataset, clustering_path, index=False)

print("Saved:", regression_path)
print("Saved:", clustering_path)

Saved: F:\DATA SCIENCE\Projects\Diamond Dynamics Price Prediction and Market Segmentation\data\processed\regression_model_input.csv
Saved: F:\DATA SCIENCE\Projects\Diamond Dynamics Price Prediction and Market Segmentation\data\processed\clustering_model_input.csv


## Split columns for preprocessors

In [9]:
regression_numeric = [col for col in regression_dataset.columns if col != "price" and regression_dataset[col].dtype != "object"]
regression_categorical = [col for col in regression_dataset.columns if col != "price" and regression_dataset[col].dtype == "object"]

clustering_numeric = [col for col in clustering_dataset.columns if clustering_dataset[col].dtype != "object"]
clustering_categorical = [col for col in clustering_dataset.columns if clustering_dataset[col].dtype == "object"]

regression_numeric, regression_categorical, clustering_numeric, clustering_categorical

(['volume', 'carat', 'depth', 'table'],
 ['clarity', 'color', 'cut'],
 ['carat',
  'depth',
  'table',
  'dimension_ratio',
  'length_width_ratio',
  'depth_pct_from_dimensions',
  'table_depth_interaction',
  'table_to_depth_ratio'],
 ['cut', 'color', 'clarity'])

## Build and fit preprocessors

In [10]:
regression_preprocessor = build_regression_preprocessor(
    numeric_cols=regression_numeric,
    categorical_cols=regression_categorical,
    project_root=project_root,
)

clustering_preprocessor = build_clustering_preprocessor(
    numeric_cols=clustering_numeric,
    categorical_cols=clustering_categorical,
    project_root=project_root,
)

X_reg_transformed = transform_with_preprocessor(
    regression_dataset.drop(columns=["price"]),
    regression_preprocessor,
    fit=True,
)

X_cluster_transformed = transform_with_preprocessor(
    clustering_dataset,
    clustering_preprocessor,
    fit=True,
)

X_reg_transformed.shape, X_cluster_transformed.shape

((53940, 24), (53940, 11))

## Save preprocessors and scalers

In [11]:
joblib.dump(
    regression_preprocessor.named_transformers_["num"].named_steps["imputer"],
    resolve_project_path(project_root, "artifacts/preprocessing/numeric_imputer.pkl"),
)

joblib.dump(
    clustering_preprocessor.named_transformers_["cat"].named_steps["encoder"],
    resolve_project_path(project_root, "artifacts/preprocessing/categorical_encoder.pkl"),
)

joblib.dump(
    regression_preprocessor,
    resolve_project_path(project_root, "artifacts/preprocessing/regression_preprocessor.pkl"),
)

joblib.dump(
    clustering_preprocessor,
    resolve_project_path(project_root, "artifacts/preprocessing/clustering_preprocessor.pkl"),
)

scaler_regression = fit_numeric_scaler(regression_dataset, regression_numeric)
scaler_clustering = fit_numeric_scaler(clustering_dataset, clustering_numeric)

joblib.dump(
    scaler_regression,
    resolve_project_path(project_root, "artifacts/preprocessing/scaler_regression.pkl"),
)

joblib.dump(
    scaler_clustering,
    resolve_project_path(project_root, "artifacts/preprocessing/scaler_clustering.pkl"),
)

print("Saved preprocessors and scalers.")

Saved preprocessors and scalers.


## Update final report

In [14]:
feature_doc_df = pd.DataFrame({
    "feature": [],
    "feature_type": [],
    "formula": [],
    "reason": [],
    "use_for_regression": [],
    "use_for_clustering": [],
    "notes": [],
})

report_text = build_feature_engineering_report(
    df=df_fe,
    feature_doc_df=feature_doc_df,
    usd_inr_rate=np.nan,
    selected_regression_features=regression_selection["selected_features"],
    selected_clustering_features=clustering_dataset.columns.tolist(),
)

save_text_file(
    report_text,
    resolve_project_path(project_root, "reports/feature_engineering_report.md"),
)

print("Updated report saved.")

Updated report saved.
